In [2]:
import pandas as pd

data = pd.read_csv("train.csv")

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
# Drop useless columns
data = data.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [5]:
# Split data into X, and y
X = data.drop(columns=["Survived"])
y = data["Survived"]

X.info()
print("\n")
y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    object 
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Embarked  889 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 48.9+ KB


<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Survived
Non-Null Count  Dtype
--------------  -----
891 non-null    int64
dtypes: int64(1)
memory usage: 7.1 KB


In [6]:
# Split X and y into train, cv, test
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state = 42)
X_cv, X_test, y_cv, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state = 42)

# Check
print(X_train.shape)
print(X_cv.shape)
print(X_test.shape)
print(y_train.shape, y_cv.shape, y_train.shape, sep = "\n")

(534, 7)
(178, 7)
(179, 7)
(534,)
(178,)
(534,)


In [7]:
# Find missing values
print(f"X_train:\n{X_train.isnull().any()}\n")
print(f"X_cv:\n{X_cv.isnull().any()}\n")
print(f"X_test:\n{X_test.isnull().any()}\n")

X_train:
Pclass      False
Sex         False
Age          True
SibSp       False
Parch       False
Fare        False
Embarked    False
dtype: bool

X_cv:
Pclass      False
Sex         False
Age          True
SibSp       False
Parch       False
Fare        False
Embarked     True
dtype: bool

X_test:
Pclass      False
Sex         False
Age          True
SibSp       False
Parch       False
Fare        False
Embarked     True
dtype: bool



In [8]:
# Fix missing values under X_train: 1) Age
age_median = X_train["Age"].median()

X_train["Age"] = X_train["Age"].fillna(age_median)
X_cv["Age"] = X_cv["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)

# Check
print(f"X_train:{X_train["Age"].isnull().any()}")
print(f"X_cv:{X_cv["Age"].isnull().any()}")
print(f"X_test:{X_test["Age"].isnull().any()}")

X_train:False
X_cv:False
X_test:False


In [9]:
# Fix missing values under X_train: 2) Embarked
# Embarked has S C Q, so we will use mode() -> most frequent elem

emb_freq = X_train["Embarked"].mode()[0]

X_cv["Embarked"] = X_cv["Embarked"].fillna(emb_freq)
X_test["Embarked"] = X_test["Embarked"].fillna(emb_freq)


# Check
print(f"X_train:{X_train["Embarked"].isnull().any()}")
print(f"X_cv:{X_cv["Embarked"].isnull().any()}")
print(f"X_test:{X_test["Embarked"].isnull().any()}")

X_train:False
X_cv:False
X_test:False


In [10]:
# Embarked having values like S C Q , we need to change them into 0 1 2
X_train = pd.get_dummies(X_train, columns=["Embarked"])
X_cv = pd.get_dummies(X_cv, columns=["Embarked"])
X_test = pd.get_dummies(X_test, columns=["Embarked"])

In [11]:
# Check
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
570,2,male,62.0,0,0,10.5000,False,False,True
787,3,male,8.0,4,1,29.1250,False,True,False
74,3,male,32.0,0,0,56.4958,False,False,True
113,3,female,20.0,1,0,9.8250,False,False,True
635,2,female,28.0,0,0,13.0000,False,False,True


In [12]:
# Change colums Sex male and female into 0 and 1
X_train["Sex"] = X_train["Sex"].map({"male": 0, "female": 1})
X_cv["Sex"] = X_cv["Sex"].map({"male": 0, "female": 1})
X_test["Sex"] = X_test["Sex"].map({"male": 0, "female": 1})

In [13]:
# Check
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
570,2,0,62.0,0,0,10.5000,False,False,True
787,3,0,8.0,4,1,29.1250,False,True,False
74,3,0,32.0,0,0,56.4958,False,False,True
113,3,1,20.0,1,0,9.8250,False,False,True
635,2,1,28.0,0,0,13.0000,False,False,True


In [27]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

param_grid = {
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200, 300],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [1, 5, 10]
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=30,
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)

{'subsample': 0.7, 'reg_lambda': 1, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
0.8464821019220597
